In [1]:
using CovidSim_ilm

In [2]:
using StatsBase
using TypedTables
using BenchmarkTools
using PrettyPrint

In [3]:
cd(joinpath(homedir(),"Dropbox/Online Coursework/Covid/ilm-src"))

# Create a seed case

In [4]:
seed_1_6 = seed_case_gen(1, [0,3,3,0,0], 1, nil, agegrps)

(::CovidSim_ilm.var"#scase#59"{Int64, Vector{Int64}, Int64, Int64, Vector{Int64}}) (generic function with 1 method)

# Run a simulation for 40 days

In [6]:
result_dict, env, series = run_a_sim(40, 38015, showr0=false, silent=true, spreadcases=[], runcases=[seed_1_6]);

*** seed day 1 locale 38015....
(sprtime, trtime, histtime) = (0.057015914, 0.16073597799999992, 0.46145876800000024)


In [7]:
keys(result_dict)

KeySet for a Dict{String, Any} with 4 entries. Keys:
  "geo"
  "sp"
  "dat"
  "dt_dict"

In [8]:
dectrees = result_dict["dt_dict"]

Dict{String, Dict{Int64, V} where V} with 4 entries:
  "lags"      => Dict(5=>[5, 9, 14, 25, 14, 9, 14, 25, 9, 19], 4=>[5, 9, 14, 25…
  "decpoints" => Dict(5=>[5, 9, 14, 25, 19], 4=>[5, 9, 14, 25, 19], 2=>[5, 9, 1…
  "dt"        => Dict{Int64, OrderedCollections.OrderedDict{Any, Any}}(5=>Order…
  "fromconds" => Dict(5=>[5, 6, 7, 8, 6, 5, 8, 7, 7, 8], 4=>[5, 6, 7, 8, 6, 5, …

In [9]:
popdat = result_dict["dat"]["popdat"]

Dict{Int64, Table{NamedTuple{(:status, :agegrp, :cond, :lag, :recov_day, :dead_day, :cluster, :vax, :vax_day, :test, :test_day, :quar, :quar_day), Tuple{Int64, Int64, Int64, Int64, Int64, Int64, Int64, Bool, Int64, Bool, Int64, Bool, Int64}}, 1, NamedTuple{(:status, :agegrp, :cond, :lag, :recov_day, :dead_day, :cluster, :vax, :vax_day, :test, :test_day, :quar, :quar_day), Tuple{Vector{Int64}, Vector{Int64}, Vector{Int64}, Vector{Int64}, Vector{Int64}, Vector{Int64}, Vector{Int64}, BitVector, Vector{Int64}, BitVector, Vector{Int64}, BitVector, Vector{Int64}}}}} with 1 entry:
  38015 => Table with 13 columns and 95626 rows:…

### Check allocations of transition! function

In [23]:
@time transition!(popdat, 38015, dectrees)

  0.077162 seconds (108.26 k allocations: 6.717 MiB)


(2.46 k allocations: 87.141 KiB): based on number of times through the loop

what's the cost per iteration?

In [11]:
dectree = dectrees["dt"] # before starting the loop--once per function call

Dict{Int64, OrderedCollections.OrderedDict{Any, Any}} with 5 entries:
  5 => OrderedCollections.OrderedDict(25=>Dict{Any, Any}(7=>Dict{String, Vector…
  4 => OrderedCollections.OrderedDict(25=>Dict{Any, Any}(7=>Dict{String, Vector…
  2 => OrderedCollections.OrderedDict(25=>Dict{Any, Any}(7=>Dict{String, Vector…
  3 => OrderedCollections.OrderedDict(25=>Dict{Any, Any}(7=>Dict{String, Vector…
  1 => OrderedCollections.OrderedDict(25=>Dict{Any, Any}(7=>Dict{String, Vector…

In [12]:
@time dectree_a1 = dectree[1]  # within the loop; no allocations ~ 5 ns

  0.000006 seconds


OrderedCollections.OrderedDict{Any, Any} with 5 entries:
  25 => Dict{Any, Any}(7=>Dict{String, Vector{T} where T}("probs"=>[0.976, 0.02…
  19 => Dict{Any, Any}(8=>Dict{String, Vector{T} where T}("probs"=>[0.891, 0.10…
  14 => Dict{Any, Any}(6=>Dict{String, Vector{T} where T}("probs"=>[1.0], "bran…
  9  => Dict{Any, Any}(5=>Dict{String, Vector{T} where T}("probs"=>[0.9, 0.1], …
  5  => Dict{Any, Any}(5=>Dict{String, Vector{T} where T}("probs"=>[0.4, 0.5, 0…

In [13]:
pprint(dectree_a1)

{
  25 : {
         7 : {
               "probs" : [0.976, 
                          0.024],
               "branches" : [
                              {
                                "tocond" : 3,
                                "next" : (0, 
                                          0),
                                "pr" : 0.976,
                              },
                              {
                                "tocond" : 4,
                                "next" : (0, 
                                          5),
                                "pr" : 0.024,
                              },
                            ],
               "outcomes" : [3, 4],
             },
         8 : {
               "probs" : [0.91, 0.09],
               "branches" : [
                              {
                                "tocond" : 3,
                                "next" : (0, 
                                          0),
                                "pr" : 0.

In [18]:
node = dectree[1][14][7] # once per iteration

Dict{String, Vector{T} where T} with 3 entries:
  "probs"    => [0.85, 0.12, 0.03]
  "branches" => Dict{Any, Any}[Dict("tocond"=>3, "next"=>(0, 0), "pr"=>0.85), D…
  "outcomes" => [3, 7, 8]

In [16]:
@btime $dectree[1][14][7]["probs"]

  92.342 ns (0 allocations: 0 bytes)


3-element Vector{Float64}:
 0.85
 0.12
 0.03

##### allocations: (1 allocation: 32 bytes) or 0 allocs: 0 bytes if empty

In [19]:
typeof(node)

Dict{String, Vector{T} where T}

In [20]:
@time node["probs"]

  0.000007 seconds


3-element Vector{Float64}:
 0.85
 0.12
 0.03

In [21]:
@time node["outcomes"]

  0.000006 seconds


3-element Vector{Int64}:
 3
 7
 8

#### transition! is not bad for allocations, but there are a lot of iterations in a simulation
It would be a fair amount of work to eliminate the allocation completely.

## try a different data structure

In [70]:
holder = Dict(14 => Dict(6=>Dict(:probs=>[1.0], :outcomes=>[3]),
                         7=>Dict(:probs=>[0.85, 0.12, 0.03], :outcomes=>[3,7,8]),
                         8=>Dict(:probs=>[0.692, 0.302, 0.006], :outcomes=>[3,8,4])), 
               9 => Dict(5=>Dict(:probs=>[.9, .1], :outcomes=>[3,7]),
                         6=>Dict(:probs=>[1.0], :outcomes=>[6]),
                         7=>Dict(:probs=>[.95, .05], :outcomes=>[7,8])))

Dict{Int64, Dict{Int64, Dict{Symbol, Vector{T} where T}}} with 2 entries:
  9  => Dict(5=>Dict(:outcomes=>[3, 7], :probs=>[0.9, 0.1]), 6=>Dict(:outcomes=…
  14 => Dict(6=>Dict(:outcomes=>[3], :probs=>[1.0]), 7=>Dict(:outcomes=>[3, 7, …

In [79]:
choice = CovidSim_ilm.categorical_sim(holder[14][7][:probs]) # no allocations here

1

In [78]:
tocond = holder[14][7][:outcomes][choice] # no allocation here

3

In [106]:
yaml_str = "1:                                         
  5:
    5:                                     
      probs: [0.4, 0.5, 0.1]
      outcomes: [5, 6, 7]
  9:
    5:                                     
      probs: [0.9, 0.1]
      outcomes: [0,7]
    6:   
      probs: [1.0]
      outcomes: [6]                                 
    7:   
      probs: [0.95, 0.05]                                   
      outcomes: [7,8]"

"1:                                         \n  5:\n    5:                                     \n      probs: [0.4, 0.5, 0.1]\n      outcomes: [5, 6, 7]\n  9:\n    5:                                     \n      probs: [0.9, 0.1]\n      outcomes: [0,7]\n    6:   \n      probs: [1.0]\n      outcomes: [6]                                 \n    7:   \n      probs: [0.95, 0.05]                                   \n      outcomes: [7,8]"

In [104]:
using YAML

In [107]:
YAML.load(yaml_str)

Dict{Any, Any} with 1 entry:
  1 => Dict{Any, Any}(5=>Dict{Any, Any}(5=>Dict{Any, Any}("probs"=>[0.4, 0.5, 0…

## Create trees from YAML file with different structure

In [110]:
dtfilename="../parameters/dec_tree_all_25.yml"

"../parameters/dec_tree_all_25.yml"

In [111]:
trees = YAML.load_file(dtfilename)
    # next: change 2nd level keys from 2 item array{Int} [9, 5] to Tuple{Int, Int} (9,5)
    trees = Dict(i => Dict(Tuple(k)=>trees[i][k] for k in keys(trees[i])) for i in keys(trees))

    # next: change the type of next node item from array{Int} [25, 8] to Tuple{Int, Int} (25, 8)
    for agegrp in agegrps
        for (k,v) in trees[agegrp]
           for item in v
                item["next"] = Tuple(item["next"])
            end
        end
    end

In [112]:
pprint(trees)

{
  5 : {
        (5, 5) : [
                   {
                     "tocond" : 5,
                     "next" : (9, 
                               5),
                     "pr" : 0.1,
                   },
                   {
                     "tocond" : 6,
                     "next" : (9, 
                               6),
                     "pr" : 0.5,
                   },
                   {
                     "tocond" : 7,
                     "next" : (9, 
                               7),
                     "pr" : 0.4,
                   },
                 ],
        (9, 6) : [
                   {
                     "tocond" : 6,
                     "next" : (14, 
                               6),
                     "pr" : 0.4,
                   },
                   {
                     "tocond" : 7,
                     "next" : (14, 
                               7),
                     "pr" : 0.6,
                   },
                 ],
     

In [113]:
newdict = Dict()
for agegrp in agegrps
    newdict[agegrp] = Dict()
    for node in keys(trees[agegrp])
        a = node[1]
        b = node[2]
            probs = [branch["pr"] for branch in trees[agegrp][node]]
            outcomes = [branch["tocond"] for branch in trees[agegrp][node]]
            branches = [branch for branch in trees[agegrp][node]]
        if haskey(newdict[agegrp], a)
            newdict[agegrp][a][b] = Dict("probs"=>probs, "outcomes"=>outcomes, "branches"=>branches)
        else
            newdict[agegrp][a]=Dict()
            newdict[agegrp][a][b] = Dict("probs"=>probs, "outcomes"=>outcomes, "branches"=>branches)
        end
    end
end
newdict = Dict(i=>sort(newdict[i], rev=true) for i in agegrps) 

Dict{Int64, OrderedCollections.OrderedDict{Any, Any}} with 5 entries:
  5 => OrderedCollections.OrderedDict(25=>Dict{Any, Any}(7=>Dict{String, Vector…
  4 => OrderedCollections.OrderedDict(25=>Dict{Any, Any}(7=>Dict{String, Vector…
  2 => OrderedCollections.OrderedDict(25=>Dict{Any, Any}(7=>Dict{String, Vector…
  3 => OrderedCollections.OrderedDict(25=>Dict{Any, Any}(7=>Dict{String, Vector…
  1 => OrderedCollections.OrderedDict(25=>Dict{Any, Any}(7=>Dict{String, Vector…

In [114]:
newdict[5]

OrderedCollections.OrderedDict{Any, Any} with 5 entries:
  25 => Dict{Any, Any}(7=>Dict{String, Vector{T} where T}("probs"=>[0.682, 0.31…
  19 => Dict{Any, Any}(8=>Dict{String, Vector{T} where T}("probs"=>[0.49, 0.24,…
  14 => Dict{Any, Any}(6=>Dict{String, Vector{T} where T}("probs"=>[0.7, 0.3], …
  9  => Dict{Any, Any}(5=>Dict{String, Vector{T} where T}("probs"=>[0.5, 0.5], …
  5  => Dict{Any, Any}(5=>Dict{String, Vector{T} where T}("probs"=>[0.1, 0.5, 0…

In [115]:
newdict[5][14]

Dict{Any, Any} with 3 entries:
  6 => Dict{String, Vector{T} where T}("probs"=>[0.7, 0.3], "branches"=>Dict{An…
  7 => Dict{String, Vector{T} where T}("probs"=>[0.7, 0.1, 0.2], "branches"=>Di…
  8 => Dict{String, Vector{T} where T}("probs"=>[0.12, 0.67, 0.21], "branches"=…

In [116]:
newdict[5][14][6]

Dict{String, Vector{T} where T} with 3 entries:
  "probs"    => [0.7, 0.3]
  "branches" => Dict{Any, Any}[Dict("tocond"=>3, "next"=>(0, 0), "pr"=>0.7), Di…
  "outcomes" => [3, 7]

In [120]:
@btime $newdict[5][14][6]["probs"]

  90.069 ns (0 allocations: 0 bytes)


2-element Vector{Float64}:
 0.7
 0.3

test it...

In [215]:
categorical_sim = CovidSim_ilm.categorical_sim

categorical_sim (generic function with 4 methods)

In [444]:
function partial_transition(tree, agegrp, lag_p, cond_p, foo=0)
    # node = get_node2(tree[agegrp], lag_p, cond_p)
    # if isempty(node) # no transition for this person based on lag and condition
    if !haskey(tree, lag_p) || !haskey(tree[lag_p], cond_p)
        # @assert p_tup.lag < laglim "Person made it to last day and was not removed:\n     $p_tup\n"
        foo += 1
    else  # change of the person p's state--a transition
        node = tree[agegrp][lag_p][cond_p]
        choice = categorical_sim(node["probs"]) # rand(Categorical(node["probs"])) # which branch...?
        tocond = node["outcomes"][choice]
        return tocond
    end
end

partial_transition (generic function with 2 methods)

In [445]:
agegrp = 1; lag_p = 14; cond_p = 6

6

In [446]:
@btime partial_transition($newdict, $agegrp, $lag_p, $cond_p)

  10.336 ns (0 allocations: 0 bytes)


1

In [440]:
180 * 1000 * (9 * 1e-9)

0.0016200000000000001

In [404]:
isa(6, Dict)

false

In [279]:
@btime newdict[agegrp][lag_p][cond_p];

  89.634 ns (0 allocations: 0 bytes)


### Check allocations of spreading (2 functions)

In [50]:
@time spread!(popdat, 38015, [], env) # returns (n_spreaders, n_contacts, n_touched, n_newly_infected)

  0.005223 seconds (1.29 k allocations: 899.457 KiB, 69.25% compilation time)


(576, 987, 532, 112)

not terrible: 0.005223 seconds (1.29 k allocations: 899.457 KiB, 69.25% compilation time)

we could pre-allocate spread_idx and contactable_idx at the pop size. would need signal value for the effective
end of the valid part of the array.